# Esplorazione — VAST Challenge 2025, Mini-Challenge 3

Questo notebook esplora i dati **domanda per domanda** (Q1–Q4 del foglio risposte). Per ogni figura trovi:
1. **Domanda** — cosa vogliamo sapere;
2. **Come leggerlo** — come si legge il grafico, in parole semplici;
3. **Codifica** — perché è fatto così (tipi di dato e regole del corso, per il report);
4. **Cosa si vede** — l'osservazione;
5. dove serve, una **tabella di messaggi** che fa da prova: il grafo è un'interpretazione, il testo è la prova.

### Regole visive valide in tutto il notebook
- **Colore = gruppo dell'attore**, sempre lo stesso in ogni grafico:
  <span style="color:#D55E00">■ Operazione Nemo Reef</span> ·
  <span style="color:#009E73">■ Controllo e istituzioni</span> ·
  <span style="color:#CC79A7">■ Alias</span> ·
  <span style="color:#0072B2">■ Investigatori</span> ·
  <span style="color:#aaaaaa">■ Altri</span>
- **Forma = tipo di attore**: ● persona · ▲ barca · ■ organizzazione · ◆ luogo · ✚ gruppo generico.
- **Scala magma = quantità** (conteggi, quote): dal giallo chiaro (pochi) al viola scuro (tanti). È una scala sequenziale a luminosità monotona e percettivamente uniforme (S07, S06): più scuro = di più. L'ho scelta perché, a differenza di viridis, non contiene il verde e il blu dei gruppi (verificato anche simulando il daltonismo). Le barre di quantità senza gruppo sono viola scuro; il grigio chiaro resta solo per lo sfondo / "altro".
- **Linee tratteggiate** = date chiave (annuncio e inizio della chiusura di Nemo Reef).

### Piccolo glossario
- **Gruppo**: insieme di attori che *si scrivono soprattutto tra loro*, calcolato dai messaggi (algoritmo di Louvain). Non è un giudizio: è chi parla con chi.
- **Matrice**: tabella colorata; ogni riga e ogni colonna è un attore, la cella dice quanti messaggi ci sono tra i due. Più scura = più messaggi.
- **Heatmap**: come la matrice, ma su righe e colonne ci sono altre cose (es. giorni e ore).
- **Firma**: il nome con cui chi scrive si presenta nel testo ("Mako here", "this is Nadia").
- **Menzione**: un nome citato nel testo di un messaggio di altri.
- **"A mette in moto B"**: A scrive a B e, entro 15 minuti, B scrive a qualcun altro. Esempio: Neptune → Remora "carichiamo alle 0600", poi Remora → Small Fry "prepara la squadra per le 0600". Contarlo dice chi dà input a chi (Q1c).
- **Arco inferito**: collegamento che lo stagista di Clepper ha *dedotto*, non letto esplicitamente (2/3 del grafo).

In [1]:
import re
from pathlib import Path

import altair as alt
import networkx as nx
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
CLEAN, PROC = ROOT / "data" / "clean", ROOT / "data" / "processed"
FIG = ROOT / "reports" / "figures" / "esplorazione"
FIG.mkdir(parents=True, exist_ok=True)
alt.data_transformers.disable_max_rows()
pd.set_option("display.max_colwidth", None)

m = pd.read_csv(CLEAN / "messages_analysis.csv", parse_dates=["timestamp"]).sort_values("timestamp").reset_index(drop=True)
m_all = pd.read_csv(CLEAN / "messages.csv", parse_dates=["timestamp"])
ev = pd.read_csv(CLEAN / "events.csv", parse_dates=["timestamp"])
rel = pd.read_csv(CLEAN / "relationships.csv")
act = pd.read_csv(CLEAN / "entity_activity.csv")
ee = pd.read_csv(PROC / "event_entities.csv", parse_dates=["timestamp"])
kind = act.set_index("name").sub_type
NAMES = sorted(act.name, key=len, reverse=True)

# ---- Gruppi: Louvain sui messaggi puliti, poi nome parlante in base a un membro noto.
pairs = m[m.sender_clean != m.receiver_clean].groupby(["sender_clean", "receiver_clean"]).size()
G = nx.Graph()
for (a, b), w in pairs.items():
    G.add_edge(a, b, weight=G.get_edge_data(a, b, {"weight": 0})["weight"] + w)
comms = nx.community.louvain_communities(G, weight="weight", seed=1)
deg = dict(G.degree(weight="weight"))
ANCHOR = {"Mako": "Operazione Nemo Reef", "Oceanus City Council": "Controllo e istituzioni",
          "Mrs. Money": "Alias", "Clepper Jensen": "Investigatori"}
GROUPS = ["Operazione Nemo Reef", "Controllo e istituzioni", "Alias", "Investigatori", "Altri"]
GROUP_OF = {}
for c in comms:
    label = next(v for k, v in ANCHOR.items() if k in c)
    GROUP_OF.update({n: label for n in c})
group = lambda name: GROUP_OF.get(name, "Altri")

# ---- Palette di progetto (design-rules.md): colore = gruppo, forma = tipo, magma = quantità.
GROUP_COLOR = alt.Scale(domain=GROUPS, range=["#D55E00", "#009E73", "#CC79A7", "#0072B2", "#cfcfcf"])
TYPE_SHAPE = alt.Scale(domain=["Person", "Vessel", "Organization", "Location", "Group"],
                       range=["circle", "triangle-up", "square", "diamond", "cross"])
GRAY, LIGHT, INK = "#51127c", "#d9d9d9", "#333333"  # GRAY = tinta scura di magma per barre di quantità
SEQ = dict(scheme="magma", reverse=True)  # sequenziale, luminosità monotona: più scuro = più messaggi
WEEK_COLOR = alt.Scale(domain=["W1", "W2"], range=["#b4a7d6", "#51127c"])
color_group = lambda field="gruppo": alt.Color(f"{field}:N", scale=GROUP_COLOR, title="gruppo")
shape_type = lambda field="tipo": alt.Shape(f"{field}:N", scale=TYPE_SHAPE, title="tipo di attore")

KEY_DATES = pd.DataFrame({"data": pd.to_datetime(["2040-10-10 09:54", "2040-10-11 06:00"]),
                          "evento": ["annuncio chiusura", "chiusura Nemo Reef"]})


def date_rules():
    rules = alt.Chart(KEY_DATES).mark_rule(strokeDash=[4, 3], color="#888").encode(x="data:T")
    left = alt.Chart(KEY_DATES.iloc[[0]]).mark_text(align="right", dx=-4, dy=-4, fontSize=10, color="#666",
                                                    baseline="bottom").encode(x="data:T", y=alt.value(0), text="evento:N")
    right = alt.Chart(KEY_DATES.iloc[[1]]).mark_text(align="left", dx=4, dy=-4, fontSize=10, color="#666",
                                                     baseline="bottom").encode(x="data:T", y=alt.value(0), text="evento:N")
    return rules + left + right


def save(chart, name):
    chart = (chart.configure(font="Helvetica Neue")
             .configure_view(strokeWidth=0)
             .configure_axis(labelFontSize=11, titleFontSize=11, titleFontWeight="normal", titleColor="#555",
                             labelColor="#444", gridColor="#ececec", domainColor="#999", tickColor="#999")
             .configure_header(labelFontSize=12, labelColor=INK, labelFontWeight=600, titleFontWeight="normal")
             .configure_legend(labelFontSize=11, titleFontSize=11, titleFontWeight="normal", titleColor="#555")
             .configure_title(fontSize=15, fontWeight=600, anchor="start", color=INK,
                              subtitleFontSize=12, subtitleColor="#666", subtitlePadding=4, offset=12))
    chart.save(FIG / f"{name}.png", scale_factor=2)
    return chart


def show(df, cols=("timestamp", "sender_clean", "receiver_clean", "content"), n=None):
    """Tabella di messaggi leggibile: testo intero, allineato a sinistra."""
    t = df[list(cols)].head(n) if n else df[list(cols)]
    t = t.rename(columns={"timestamp": "quando", "sender_clean": "da", "receiver_clean": "a", "content": "testo"})
    if "quando" in t:
        t["quando"] = t.quando.dt.strftime("%d %b %H:%M")
    display(t.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))


m["gruppo_mitt"] = m.sender_clean.map(group)
m["gruppo_dest"] = m.receiver_clean.map(group)
m["week"] = (m.timestamp >= "2040-10-08").map({False: "W1", True: "W2"})
m["day"] = m.timestamp.dt.strftime("%d %b")
pd.Series(GROUP_OF).rename("gruppo").groupby(lambda x: GROUP_OF[x]).size()

Alias                       8
Controllo e istituzioni    15
Investigatori               2
Operazione Nemo Reef       15
Name: gruppo, dtype: int64

## 0. Mappa di partenza: chi è chi

**Domanda.** Quali attori ci sono e in quale gruppo stanno?

**Come leggerlo.** Ogni colonna è un gruppo. Dentro, gli attori sono ordinati dal più attivo (in alto) al meno attivo; la forma del simbolo dice se è una persona, una barca, un'organizzazione o un luogo. Il numero è quanti messaggi ha inviato o ricevuto.

<sub>Codifica: gruppo **N** → colonne e colore; tipo **N** → forma; messaggi **Q** → ordine verticale ed etichetta.</sub>

In [2]:
roster = act[(act.msg_sent + act.msg_received) > 0].assign(
    messaggi=lambda d: d.msg_sent + d.msg_received, gruppo=lambda d: d.name.map(group), tipo=lambda d: d.sub_type)
roster["rank"] = roster.groupby("gruppo").messaggi.rank(ascending=False, method="first")
roster["etichetta"] = roster.name + "  (" + roster.messaggi.astype(str) + ")"
base = alt.Chart().encode(y=alt.Y("rank:O", axis=None))
pts = base.mark_point(filled=True, size=90).encode(x=alt.value(8), color=color_group(), shape=shape_type(),
                                                    tooltip=["name", "tipo", "messaggi"])
txt = base.mark_text(align="left", dx=18, fontSize=11, color=INK).encode(x=alt.value(0), text="etichetta:N")
m00 = alt.layer(pts, txt, data=roster).properties(width=170, height=alt.Step(17)).facet(
    column=alt.Column("gruppo:N", sort=GROUPS, title=None)).properties(title=alt.Title(
    "Quattro gruppi: l'operazione, chi la controlla, gli alias e gli investigatori",
    subtitle="attori ordinati per numero di messaggi (inviati + ricevuti); colore = gruppo, forma = tipo di attore"))
save(m00, "m00_chi_e_chi")

alt.FacetChart(...)

**Cosa si vede.** L'**operazione Nemo Reef** riunisce barche (Mako, Remora, Neptune), V. Miesel Shipping, Davis, Rodriguez, **Nadia Conti** e lo staff di Sailor Shift. Il gruppo **controllo e istituzioni** contiene il Council, i porti, Green Guardians e le sue barche, e anche **Liam Thorne**. Gli **alias** (Mrs. Money, The Intern, The Lookout, Boss, The Middleman, The Accountant) parlano soprattutto tra loro, con Sam e Kelly. Gli **investigatori** sono solo Clepper e Miranda.

**Date chiave della storia** (dagli eventi registrati e dai messaggi):

| Data | Cosa succede |
|---|---|
| 1 ott | primi avvertimenti a Remora e Mako vicino a Nemo Reef |
| 5 ott | Nadia entra in scena: cancella i piani ad Haacklee Harbor e chiede di distruggere i documenti |
| 8 ott | rendezvous (incontro in mare) di Mako, Remora e Neptune a Nemo Reef |
| 10 ott 09:54 | annuncio: Nemo Reef chiusa per 3 giorni |
| 11 ott 06:00 | inizio della chiusura con zona di rispetto di 500 m |

---
## Q1 — Quando si comunica, e chi influenza chi

> *Clepper ha notato che i messaggi arrivano spesso alla stessa ora. Trovare i pattern giornalieri, come cambiano nelle due settimane, e chi ha influenza su un'entità.*

**Domanda.** A che ora arrivano i messaggi, giorno per giorno?

**Come leggerlo.** Righe = giorni, colonne = ore del giorno (tutte le 24). Ogni cella è tanto più scura (dal giallo al viola) quanti più messaggi ci sono in quell'ora; il numero è il conteggio. Le celle bianche = nessun messaggio.

<sub>Codifica: giorno **O**, ora **O** (ciclica), messaggi **Q** → heatmap (due ordinali + un conteggio, S06 p.12) con scala sequenziale magma (luminosità monotona); tutte le 24 ore per mostrare il vuoto notturno.</sub>

In [3]:
grid = pd.MultiIndex.from_product([m.day.unique(), range(24)], names=["day", "hour"])
hm = m.groupby(["day", "hour"]).size().reindex(grid, fill_value=0).reset_index(name="n")
b = alt.Chart(hm).encode(x=alt.X("hour:O", title="ora del giorno", axis=alt.Axis(labelAngle=0)),
                         y=alt.Y("day:O", title=None, sort=list(m.day.unique())))
q1a = (b.mark_rect().encode(color=alt.condition("datum.n == 0", alt.value("#f5f5f5"),
                                               alt.Color("n:Q", scale=alt.Scale(**SEQ, domainMin=1), legend=None)),
                            tooltip=["day", "hour", "n"])
       + b.mark_text(fontSize=9).encode(text=alt.condition("datum.n > 0", "n:Q", alt.value("")),
                                        color=alt.condition("datum.n > 12", alt.value("white"), alt.value(INK)))
       ).properties(width=alt.Step(22), height=alt.Step(18), title=alt.Title(
    "La radio si usa solo tra le 8 e le 14, con un picco diverso ogni giorno",
    subtitle="messaggi per giorno e ora"))
save(q1a, "q1a_heatmap_giorno_ora")

alt.LayerChart(...)

**Cosa si vede.** Nessun messaggio prima delle 8 o dopo le 14 (un solo caso alle 14): le comunicazioni hanno un **orario fisso**, come un turno. L'ora più trafficata però cambia: alle 8 l'1 e il 4 ottobre, alle 10 il 3, il 5 e l'11, alle 12 il 2, il 13 e il 14. Il massimo (24 messaggi) è il **5 ottobre alle 10**, il giorno in cui Nadia cancella i piani ad Haacklee Harbor.

**Domanda.** Il ritmo della giornata cambia tra la prima e la seconda settimana?

**Come leggerlo.** Due linee sullo stesso asse: viola chiaro = prima settimana (1–7 ott), viola scuro = seconda (8–14 ott). L'altezza è il numero medio di messaggi al giorno in quell'ora.

<sub>Codifica: ora **O**, settimana **N** (2 classi), messaggi medi **Q** → linee sovrapposte: posizione su scala comune, la codifica più precisa per confrontare due profili.</sub>

In [4]:
wk = (m.groupby(["week", "hour"]).size().div(7)
      .reindex(pd.MultiIndex.from_product([["W1", "W2"], range(8, 15)], names=["week", "hour"]), fill_value=0)
      .reset_index(name="media"))
line = alt.Chart(wk).encode(
    x=alt.X("hour:O", title="ora", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("media:Q", title="messaggi medi al giorno", scale=alt.Scale(domainMin=0)),
    color=alt.Color("week:N", scale=WEEK_COLOR, title="settimana"),
    tooltip=["week", "hour", alt.Tooltip("media:Q", format=".1f")])
q1b = (line.mark_line(strokeWidth=2.5) + line.mark_circle(size=60, opacity=1)).properties(width=420, height=220, title=alt.Title(
    "Nella seconda settimana sparisce il picco delle 10: il traffico si distribuisce",
    subtitle="messaggi medi al giorno per ora; W1 = 1–7 ott, W2 = 8–14 ott"))
save(q1b, "q1b_profilo_settimane")

alt.LayerChart(...)

**Cosa si vede.** In W1 c'è un picco netto alle 10 (12 messaggi al giorno); in W2 il traffico è più piatto tra le 8 e le 12 e quasi sparisce alle 13. La giornata "tipo" cambia nel momento in cui partono i lavori a Nemo Reef (dall'8 ottobre).

### L'agenda nascosta
La radio si usa di giorno, ma nel testo si fissano appuntamenti ("meet at 0500", "tonight 2100").
Estraggo gli **orari citati** (formati `0500`, `05:00`, `5 AM`, `7PM`), escludendo codici come `#CR-7844`.

**Domanda.** A che ora avvengono le attività di cui si parla nei messaggi?

**Come leggerlo.** Ogni barra è un'ora del giorno; l'altezza dice quante volte quell'ora è citata in un messaggio. La fascia grigia chiara è l'orario in cui si usa la radio (8–14).

<sub>Codifica: ora citata **O** (ciclica, 0–23), menzioni **Q** → barre da zero; la fascia radio sulla stessa scala permette il confronto diretto.</sub>

In [5]:
TIME_RE = re.compile(
    r"(?<![#\w-])(?:(?P<hh>[01]\d|2[0-3])[:.]?(?P<mm>[0-5]\d)(?![\d%-])(?:\s*(?:hours|hrs)\b)?"
    r"|(?P<h12>1[0-2]|0?[1-9])(?::[0-5]\d)?\s*(?P<ap>[AaPp])\.?[Mm]\b)")


def mentioned_hours(text):
    return [int(t.group("hh")) if t.group("hh") else int(t.group("h12")) % 12 + (12 if t.group("ap").lower() == "p" else 0)
            for t in TIME_RE.finditer(text)]


m["ore_citate"] = m.content.map(mentioned_hours)
cit = m[m.ore_citate.str.len() > 0].explode("ore_citate").astype({"ore_citate": int})
cit["fuori_radio"] = ~cit.ore_citate.between(8, 14)
n_msg, share = m.ore_citate.str.len().gt(0).sum(), cit.fuori_radio.mean()
hist = cit.groupby("ore_citate").size().reindex(range(24), fill_value=0).rename_axis("ora").reset_index(name="n")
hx = alt.Scale(domain=[-0.5, 23.5], nice=False)
band = alt.Chart(pd.DataFrame({"a": [7.5], "b": [14.5]})).mark_rect(color="#f0f0f0").encode(x=alt.X("a:Q", scale=hx), x2="b:Q")
band_t = alt.Chart(pd.DataFrame({"x": [11], "t": ["radio attiva"]})).mark_text(baseline="top", dy=4, fontSize=11, color="#777").encode(
    x=alt.X("x:Q", scale=hx), y=alt.value(0), text="t:N")
bars = alt.Chart(hist).mark_bar(color=GRAY, width=16).encode(
    x=alt.X("ora:Q", scale=hx, title="ora citata nel testo", axis=alt.Axis(values=list(range(24)), grid=False)),
    y=alt.Y("n:Q", title="quante volte è citata"), tooltip=["ora", "n"])
q1c = (band + band_t + bars).properties(width=560, height=220, title=alt.Title(
    "Gli appuntamenti si fissano all'alba e di sera, quando la radio tace",
    subtitle=f"orari citati in {n_msg} messaggi: il {share:.0%} cade fuori dalla fascia radio (8–14)"))
save(q1c, "q1c_orari_citati")

alt.LayerChart(...)

**Cosa si vede.** L'**83%** degli orari citati è fuori dalla fascia radio: soprattutto **5–6 del mattino** e **21–22**. La radio serve a *organizzare*; le attività vere avvengono quando nessuno ascolta. Quindi i "pattern temporali" di Q1 sono due: quando si parla (8–14) e quando si agisce (alba e sera).

In [6]:
print("Esempi di appuntamenti fuori orario:")
show(cit[cit.fuori_radio].drop_duplicates("msg_id").sample(6, random_state=3).sort_values("timestamp"))

Esempi di appuntamenti fuori orario:


quando,da,a,testo
01 Oct 08:59,Himark Harbor,Mako,Harbor Master to Mako. Confirming your 07:00 departure today with Remora for Sailors Team's reconnaissance. Please remember to return before sunset and maintain distance from protected marine areas. Safe journey.
01 Oct 10:55,Mrs. Money,The Middleman,"The Middleman, it's Mrs. Money. 7PM at the cafe works perfectly. I'll bring those financial projections on the Nemo Reef opportunity. I've also identified potential funding sources that won't raise flags with regulators."
07 Oct 11:05,Mako,Remora,"Remora, Mako here. Agree on avoiding Route A. Route C makes sense for the fragile equipment transfer. Meeting Elise at eastern reef tomorrow at 0600 for payment protocols. Can you confirm your ETA at the transfer point?"
11 Oct 08:12,Mrs. Money,The Middleman,"The Middleman, Mrs. Money here. Any intel on V. Miesel's involvement? Conservation vessels are taking water samples near Nemo Reef. Need to know if they've found anything before our 0500 meeting. Time is critical."
14 Oct 09:02,Small Fry,Remora,"Remora, Small Fry here. Will meet at south dock at 0200 for equipment transfer. Navigation charts prepared and team briefed on silent protocols. Conservation vessel positions mapped for optimal routing. Ready for your instructions."
14 Oct 13:35,Knowles,Davis,"Davis, this is Knowles. Approaching southern dock for 22:00 equipment transfer as instructed by Mako. All systems operational and crew ready. Awaiting your debrief at 21:30. Maintaining security protocols."


**Domanda.** Chi organizza le attività fuori orario?

**Come leggerlo.** Ogni barra è un attore; la lunghezza è il numero di suoi messaggi che fissano un orario prima delle 8 o dopo le 14. Il colore è il gruppo.

<sub>Codifica: attore **N**, messaggi **Q**, gruppo **N** → barre ordinate da zero, colore = gruppo.</sub>

In [7]:
night = cit[cit.fuori_radio].drop_duplicates("msg_id")
who = night.groupby("sender_clean").size().rename("n").reset_index().assign(gruppo=lambda d: d.sender_clean.map(group))
q1d = alt.Chart(who.nlargest(18, "n")).mark_bar().encode(
    x=alt.X("n:Q", title="messaggi che fissano un orario fuori fascia radio", axis=alt.Axis(tickMinStep=1)),
    y=alt.Y("sender_clean:N", sort="-x", title=None), color=color_group(), tooltip=["sender_clean", "gruppo", "n"],
).properties(width=360, height=alt.Step(16), title=alt.Title(
    "Le attività fuori orario le organizzano l'operazione e gli alias",
    subtitle="mittenti dei messaggi che fissano un orario prima delle 8 o dopo le 14 (primi 18)"))
save(q1d, "q1d_chi_organizza_fuori_orario")
print(who.groupby("gruppo").n.sum().sort_values(ascending=False).to_dict())

{'Operazione Nemo Reef': 75, 'Alias': 35, 'Controllo e istituzioni': 17, 'Investigatori': 2}


**Cosa si vede.** Quasi tutti i messaggi con orari notturni o all'alba vengono dall'**operazione** (Mako, Neptune, Remora, Davis, Nadia) e dagli **alias** (Mrs. Money, The Intern, The Middleman, Boss). Il gruppo di **controllo** quasi mai: le sue attività sono di giorno.

**Domanda.** Chi è presente in quali giorni?

**Come leggerlo.** Righe = attori (con almeno 4 messaggi), colonne = giorni. La cella è più scura quanti più messaggi l'attore ha scambiato quel giorno. La striscia colorata a sinistra è il gruppo. Le righe sono ordinate per "giorno medio": chi è attivo soprattutto all'inizio sta in alto, chi entra tardi in basso.

<sub>Codifica: attore **N**, giorno **O**, messaggi **Q** → heatmap con scala magma; gruppo **N** → striscia colorata; righe ordinate per giorno medio (l'ordine è una scelta di design).</sub>

In [8]:
long = pd.concat([m[["day", "sender_clean"]].set_axis(["day", "attore"], axis=1),
                  m[["day", "receiver_clean"]].set_axis(["day", "attore"], axis=1)])
pres = long.groupby(["attore", "day"]).size().reset_index(name="n")
pres = pres[pres.attore.isin(pres.groupby("attore").n.sum().loc[lambda s: s >= 4].index)]
days = list(m.day.unique())
pres["d"] = pres.day.map(days.index)
order = pres.groupby("attore").apply(lambda g: (g.d * g.n).sum() / g.n.sum()).sort_values().index.tolist()
y = alt.Y("attore:N", sort=order, scale=alt.Scale(domain=order), title=None)
strip = alt.Chart(pd.DataFrame({"attore": order}).assign(gruppo=lambda d: d.attore.map(group))).mark_rect().encode(
    y=y, color=alt.Color("gruppo:N", scale=GROUP_COLOR, legend=None), tooltip=["attore", "gruppo"]).properties(width=10, height=alt.Step(12))
heat = alt.Chart(pres).mark_rect().encode(
    x=alt.X("day:O", sort=days, title=None, axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("attore:N", sort=order, scale=alt.Scale(domain=order), axis=None),
    color=alt.Color("n:Q", scale=alt.Scale(**SEQ, domainMin=0), title="messaggi"),
    tooltip=["attore", "day", "n"]).properties(width=alt.Step(20), height=alt.Step(12))
q1e = alt.hconcat(strip, heat, spacing=2).resolve_scale(color="independent").properties(title=alt.Title(
    "Chi entra e chi esce di scena: Nadia dal 5 ottobre, The Accountant dall'11",
    subtitle="messaggi per attore e giorno; striscia = gruppo; righe ordinate dal più \"precoce\" al più \"tardivo\""))
save(q1e, "q1e_presenza_per_giorno")

/var/folders/bc/w1x0r5wd03l3mc5pwgk94nqr0000gn/T/ipykernel_70061/3335888906.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  order = pres.groupby("attore").apply(lambda g: (g.d * g.n).sum() / g.n.sum()).sort_values().index.tolist()


alt.HConcatChart(...)

**Cosa si vede.** Alcuni attori sono presenti tutto il periodo (Mako, Council, The Lookout), altri entrano o escono: **Boss** è attivo soprattutto nei primi giorni; **Nadia** entra il **5 ottobre**; **Elise** dal 7; **The Accountant** e **Knowles** solo dall'11; **Neptune** è molto attivo tra il 6 e il 12, cioè nei giorni dei lavori.

### Chi influenza chi (Q1c)
Definizione: **A mette in moto B** se A scrive a B e, entro 15 minuti, B scrive a un'altra persona.
La risposta a un messaggio arriva in mediana dopo 2 minuti, quindi 15 minuti bastano a coprire una reazione.
È un indizio, non una prova: B potrebbe scrivere ad altri per motivi suoi. Per questo sotto ci sono esempi reali.

**Domanda.** Per gli attori chiave, chi li mette in moto più spesso?

**Come leggerlo.** Un pannello per attore (in alto il nome). Ogni barra è qualcuno che gli ha scritto poco prima che lui scrivesse ad altri; la lunghezza è quante volte è successo. Colore = gruppo di chi scrive.

<sub>Codifica: attore attivato **N** (4 pannelli), attivatore **N**, conteggio **Q** → small multiples di barre ordinate con la stessa scala; colore = gruppo.</sub>

In [9]:
WIN = pd.Timedelta("15min")
chain = []
for _, x in m.iterrows():
    nxt = m[(m.timestamp > x.timestamp) & (m.timestamp <= x.timestamp + WIN)
            & (m.sender_clean == x.receiver_clean) & (m.receiver_clean != x.sender_clean)]
    if len(nxt):
        y_ = nxt.iloc[0]
        chain.append(dict(attivatore=x.sender_clean, attivato=x.receiver_clean, terzo=y_.receiver_clean,
                          t1=x.timestamp, msg1=x.content, t2=y_.timestamp, msg2=y_.content))
chain = pd.DataFrame(chain)
TARGETS = ["Nadia Conti", "Remora", "Neptune", "Mako"]
inf = chain[chain.attivato.isin(TARGETS)].groupby(["attivato", "attivatore"]).size().reset_index(name="n")
inf["gruppo"] = inf.attivatore.map(group)
inf = inf[inf.groupby("attivato").n.rank(ascending=False, method="first") <= 8]
q1f = alt.Chart(inf).mark_bar().encode(
    x=alt.X("n:Q", title="volte", axis=alt.Axis(tickMinStep=1)),
    y=alt.Y("attivatore:N", sort="-x", title=None), color=color_group(), tooltip=["attivato", "attivatore", "n"],
).properties(width=140, height=alt.Step(15)).facet(
    column=alt.Column("attivato:N", sort=TARGETS, title=None)).resolve_scale(y="independent").properties(title=alt.Title(
    "Nadia è messa in moto da Davis; le barche si passano gli ordini tra loro",
    subtitle="chi scrive a X poco prima che X scriva ad altri (entro 15 minuti); primi 8 per pannello"))
save(q1f, "q1f_chi_mette_in_moto")

alt.FacetChart(...)

**Cosa si vede.** **Nadia** viene messa in moto soprattutto da **Davis** (poi Elise, Liam Thorne): non è lei ad aprire le catene di messaggi via radio, coerente con il fatto che si organizza fuori orario. **Remora** reagisce a Mako, Neptune e Small Fry; **Neptune** a Mako, Remora e V. Miesel: dentro l'operazione gli ordini passano di barca in barca. Mako reagisce anche al gruppo di controllo (porti, Council): è la barca più "sorvegliata".

In [10]:
print("Esempi di catene che coinvolgono Nadia (primo messaggio → reazione entro 15 minuti):")
ex = chain[chain.attivato == "Nadia Conti"].head(4)
display(ex[["t1", "attivatore", "msg1", "t2", "terzo", "msg2"]].style.hide(axis="index")
        .set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))

Esempi di catene che coinvolgono Nadia (primo messaggio → reazione entro 15 minuti):


t1,attivatore,msg1,t2,terzo,msg2
2040-10-05 09:44:00,Haacklee Harbor,"Haacklee Harbor to Nadia Conti. Following your visit yesterday regarding the Nemo Reef event logistics, we've prepared the necessary documentation. Harbor staff is ready to facilitate the special access corridor arrangements as discussed. Please confirm timeline for implementation.",2040-10-05 09:49:00,Oceanus City Council,This is Nadia Conti. My cancellation was due to scheduling conflicts with our tourism development initiatives. I wasn't aware of any permit approvals. I'll submit revised documentation for alternative sustainable tourism proposals next week.
2040-10-07 11:57:00,Sailor Shifts Team,"Hi Nadia, this is the Sailor Shifts Team. Received your message about permit assistance - thank you! We urgently need to discuss tomorrow's staffing requirements. Can you confirm how many additional crew members we should bring for the setup?",2040-10-07 12:00:00,Davis,"Davis, Nadia here. Let's meet at 7PM at the marina office to review documentation. I've been working with alternative channels for permits. Bring all shipping manifests - we'll need to create a clean paper trail immediately."
2040-10-08 08:15:00,Elise,"Nadia, Elise here. Meeting at Nemo Reef 0500 tomorrow to establish payment protocols. Sam uncovered V. Miesel shipping lanes overlapping with Mako by 40%. Neptune mentioned 'underwater foundation work' - outside our original scope. Need your assessment.",2040-10-08 08:18:00,Liam Thorne,"Liam, Nadia here. Need your services urgently. Investigation brewing around Nemo Reef permits. Double your usual fee if you can ensure Harbor Master remains cooperative through next week. Meet at the usual place tomorrow, 10PM."
2040-10-08 08:24:00,Liam Thorne,"Nadia, Liam here. Meeting confirmed for tomorrow at 10PM. I've redirected Harbor Master's attention and implemented new patrol schedules that work in our favor. Council suspects nothing about Nemo Reef. Bring payment as discussed.",2040-10-08 08:25:00,Neptune,"Neptune, this is Nadia. Need clarity on 'underwater foundation work' at Nemo Reef. This extends beyond our agreed scope. Meet me at the marina tomorrow at 6AM to discuss implications and additional resource requirements."


---
## Q2 — Gruppi e temi

> *Persone e barche comunicano spesso con (o su) le stesse persone. Esistono gruppi più legati? Di cosa parla ciascuno?*

**Domanda.** Chi parla con chi? I gruppi sono separati?

**Come leggerlo.** Ogni riga è un mittente, ogni colonna un destinatario, nello stesso ordine (attori raggruppati per gruppo; la striscia colorata a sinistra dice il gruppo). Una cella colorata = ci sono messaggi tra i due; più scura = più messaggi. Se i gruppi sono veri, le celle si concentrano in **blocchi lungo la diagonale**; le celle fuori dai blocchi sono i contatti tra gruppi.

<sub>Codifica: mittente **N**, destinatario **N**, messaggi **Q** (coda lunga) → matrice di adiacenza ordinata per gruppo invece del grafo nodo-link (hairball); scala magma su radice quadrata perché pochi valori alti schiaccerebbero gli altri.</sub>

In [11]:
order = sorted(GROUP_OF, key=lambda n: (GROUPS.index(group(n)), -deg[n]))
mat = pairs.reset_index(name="n").set_axis(["mittente", "destinatario", "n"], axis=1)
sc = alt.Scale(domain=order)
strip = alt.Chart(pd.DataFrame({"attore": order}).assign(gruppo=lambda d: d.attore.map(group))).mark_rect().encode(
    y=alt.Y("attore:N", sort=order, scale=sc, title=None), color=alt.Color("gruppo:N", scale=GROUP_COLOR, legend=None), tooltip=["attore", "gruppo"]
).properties(width=10, height=alt.Step(11))
matrix = alt.Chart(mat).mark_rect().encode(
    x=alt.X("destinatario:N", sort=order, scale=sc, title="destinatario", axis=alt.Axis(labelAngle=-90)),
    y=alt.Y("mittente:N", sort=order, scale=sc, axis=None),
    color=alt.Color("n:Q", scale=alt.Scale(**SEQ, type="sqrt", domainMin=0), title="messaggi"),
    tooltip=["mittente", "destinatario", "n"]).properties(width=alt.Step(11), height=alt.Step(11))
q2a = alt.hconcat(strip, matrix, spacing=2).resolve_scale(color="independent").properties(title=alt.Title(
    "L'81% dei messaggi resta dentro il proprio gruppo: quattro blocchi lungo la diagonale",
    subtitle="matrice mittente × destinatario; striscia a sinistra = gruppo del mittente"))
save(q2a, "q2a_matrice_messaggi")
mm = mat.assign(g1=mat.mittente.map(group), g2=mat.destinatario.map(group))
print("quota di messaggi interni ai gruppi:", round(mm[mm.g1 == mm.g2].n.sum() / mm.n.sum(), 2))

quota di messaggi interni ai gruppi: 0.81


**Cosa si vede.** I quattro gruppi sono netti (81% dei messaggi interni). I contatti **tra gruppi** sono pochi ma rivelatori: **Mrs. Money e Boss (alias) scrivono a Mako, Remora, Neptune e V. Miesel**, cioè all'operazione; **The Lookout scrive alle barche ambientaliste**; porti e barche di controllo scrivono a Mako.

**Domanda.** Tra quali gruppi passano i messaggi?

**Come leggerlo.** Riassunto della matrice precedente: righe = gruppo di chi scrive, colonne = gruppo di chi riceve. Il numero è quanti messaggi; la diagonale sono i messaggi interni.

<sub>Codifica: gruppo mittente **N**, gruppo destinatario **N**, messaggi **Q** → piccola matrice con valori scritti (lettura esatta) e scala magma su radice quadrata.</sub>

In [12]:
gg = m.groupby(["gruppo_mitt", "gruppo_dest"]).size().reset_index(name="n")
gg = gg[gg.gruppo_mitt != gg.gruppo_dest]
b = alt.Chart(gg).encode(x=alt.X("gruppo_dest:N", sort=GROUPS, title="a (gruppo che riceve)", axis=alt.Axis(labelAngle=-20)),
                         y=alt.Y("gruppo_mitt:N", sort=GROUPS, title="da (gruppo che scrive)"))
q2b = (b.mark_rect(cornerRadius=2).encode(color=alt.Color("n:Q", scale=alt.Scale(**SEQ, type="sqrt", domainMin=0), legend=None),
                                          tooltip=["gruppo_mitt", "gruppo_dest", "n"])
       + b.mark_text(fontSize=12).encode(text="n:Q", color=alt.condition("datum.n > 15", alt.value("white"), alt.value(INK)))
       ).properties(width=alt.Step(110), height=alt.Step(34), title=alt.Title(
    "Controllo e operazione si scrivono molto; gli alias scrivono a entrambi",
    subtitle="messaggi tra gruppi diversi (esclusi quelli interni)"))
save(q2b, "q2b_messaggi_tra_gruppi")

alt.LayerChart(...)

**Cosa si vede.** Il ponte più forte è tra **controllo e operazione** (50 messaggi in un verso, 34 nell'altro): porti, Council e barche ambientaliste chiedono conto a Mako & co., e l'operazione risponde. Gli **alias** scrivono 14 volte al controllo (soprattutto The Lookout alle barche ambientaliste) e 7 all'operazione (Mrs. Money e Boss a Mako, Neptune, V. Miesel). Gli **investigatori** non scrivono a nessuno fuori dal loro gruppo: ascoltano soltanto.

In [13]:
print("Esempi: messaggi dagli alias all'operazione")
show(m[(m.gruppo_mitt == "Alias") & (m.gruppo_dest == "Operazione Nemo Reef")], n=6)

Esempi: messaggi dagli alias all'operazione


quando,da,a,testo
04 Oct 13:48,Boss,V. Miesel Shipping,This is Boss. Need urgent confirmation that assets are properly reallocated to our tourism ventures. Conservation scrutiny has increased dramatically. Ensure all vessel documentation shows legitimate tourist operations only. Expecting verification by tomorrow.
04 Oct 13:51,Boss,Mako,"Mako, this is Boss. Abort current mission immediately. Maintain radio silence but change course away from Nemo Reef. Conservation patrols are tracking you. Return to Himark Harbor using route B. Confirm receipt."
05 Oct 10:41,Mrs. Money,V. Miesel Shipping,This is Mrs. Money. Need immediate confirmation that our recent financial arrangements remain secure and untraceable. Conservation activity has increased around Nemo Reef. Require alternate routing options for future transactions. Respond only through encrypted channels.
07 Oct 09:31,Mrs. Money,Neptune,"Neptune, this is Mrs. Money. Confirmed for 0500 tomorrow. Will coordinate payment protocols with Mako and Remora captains. Your eastern reef route is approved - it avoids the new permit zones. Bring encryption devices for financial transfers."
07 Oct 09:38,Mrs. Money,Mako,"Mako, Mrs. Money here. My role is financial controller for this operation - I'll be handling payment distribution, equipment funding, and resource allocation. Will bring encryption protocols and transfer devices. Eastern reef route documentation prepared for your review."
08 Oct 08:34,Mrs. Money,Remora,"Remora, Mrs. Money here. Will be at Nemo Reef 0500 tomorrow to establish payment protocols with you and Mako. Bring documentation for specialized equipment. I'll handle all financial transfers and resource allocation for the operation."


**Domanda.** Che tipo di legami registrati ci sono dentro ogni gruppo?

**Come leggerlo.** Un pannello per gruppo; ogni barra è un tipo di relazione del grafo (es. *Operates* = chi gestisce una barca, *Coordinates* = chi coordina un'attività con chi, *Colleagues* = colleghi). L'ultimo pannello sono le relazioni tra gruppi diversi o con luoghi che non scrivono messaggi.

<sub>Codifica: gruppo **N** (pannelli), tipo di relazione **N** (9, troppe per il colore), conteggio **Q** → small multiples di barre con la stessa scala; colore = gruppo del pannello.</sub>

In [14]:
r = rel[~rel.is_duplicate].copy()
r["g1"], r["g2"] = r.source.map(group), r.target.map(group)
r["pannello"] = r.g1.where((r.g1 == r.g2) & (r.g1 != "Altri"), "tra gruppi / con luoghi")
rg = r.groupby(["pannello", "rel_type"]).size().reset_index(name="n")
rg["gruppo"] = rg.pannello.where(rg.pannello.isin(GROUPS), "Altri")
rel_order = rg.groupby("rel_type").n.sum().sort_values(ascending=False).index.tolist()
panels = GROUPS[:4] + ["tra gruppi / con luoghi"]
bar = alt.Chart().mark_bar().encode(x=alt.X("n:Q", title="relazioni"), y=alt.Y("rel_type:N", sort=rel_order, title=None),
                                    color=alt.Color("gruppo:N", scale=GROUP_COLOR, legend=None), tooltip=["pannello", "rel_type", "n"])
q2c = alt.layer(bar, bar.mark_text(align="left", dx=3, fontSize=10, color=INK).encode(text="n:Q"), data=rg).properties(
    width=110, height=alt.Step(16)).facet(column=alt.Column("pannello:N", sort=panels, title=None)).properties(title=alt.Title(
    "Ogni gruppo ha un legame tipico: Operates nell'operazione, Coordinates nel controllo, Colleagues tra gli alias",
    subtitle="relazioni del grafo (senza duplicati) con i due attori nello stesso gruppo"))
save(q2c, "q2c_relazioni_per_gruppo")

alt.FacetChart(...)

**Cosa si vede.** **Operazione**: soprattutto *Operates* (chi gestisce quale barca) e *Coordinates*. **Controllo**: *Coordinates* e *Reports* (rapporti al Council). **Alias**: *Colleagues*. Le relazioni *Suspicious* e *AccessPermission* stanno quasi tutte nell'ultimo pannello: sono **tra** gruppi o verso Nemo Reef.

### Di cosa parla ogni gruppo
I temi sono definiti con **parole chiave** cercate nel testo (es. *soldi* = payment, fund, money, fee…). È un'approssimazione: sotto il grafico ci sono esempi per controllare che i temi siano riconosciuti bene.

**Domanda.** Quali temi prevalgono in ogni gruppo?

**Come leggerlo.** Un pannello per gruppo; ogni barra è un tema; la lunghezza è la percentuale di messaggi del gruppo che toccano quel tema (un messaggio può toccarne più d'uno).

<sub>Codifica: gruppo **N** (pannelli), tema **N**, quota **Q** 0–100% → small multiples di barre su scala fissa 0–100%; colore = gruppo.</sub>

In [15]:
TOPICS = {
    "Nemo Reef": r"nemo reef",
    "permessi": r"permit|authoriz|clearance",
    "soldi / pagamenti": r"payment|fund|money|financ|\bfees?\b|\bpay\b",
    "attrezzature / lavori": r"equipment|foundation|construct|install|stabiliz|concrete",
    "sorveglianza / pattuglie": r"patrol|surveil|monitor|\bwatch",
    "ambiente / conservazione": r"conservation|environment|water quality|wildlife|coral",
    "evento Sailor Shift": r"sailor|video|film|concert|music|\bevent\b",
    "segretezza": r"discreet|radar|encrypt|secre|confiden|delta-3|quiet|facade",
    "chiusura dell'area": r"closure|closed|restricted",
    "natura / tempo libero": r"bird|dolphin|whale|shearwater",
}
low = m.content.str.lower()
tp = pd.DataFrame([(g, t, low[m.gruppo_mitt == g].str.contains(p).mean()) for g in GROUPS[:4] for t, p in TOPICS.items()],
                  columns=["gruppo", "tema", "quota"])
bar = alt.Chart().mark_bar().encode(
    x=alt.X("quota:Q", title="% dei messaggi", scale=alt.Scale(domain=[0, 1]), axis=alt.Axis(format="%", tickCount=3)),
    y=alt.Y("tema:N", sort=list(TOPICS), title=None), color=alt.Color("gruppo:N", scale=GROUP_COLOR, legend=None),
    tooltip=["gruppo", "tema", alt.Tooltip("quota:Q", format=".0%")])
q2d = alt.layer(bar, bar.mark_text(align="left", dx=3, fontSize=9, color=INK).encode(text=alt.Text("quota:Q", format=".0%")),
                data=tp).properties(width=120, height=alt.Step(16)).facet(
    column=alt.Column("gruppo:N", sort=GROUPS, title=None)).properties(title=alt.Title(
    "Soldi per gli alias, permessi e lavori per l'operazione, sorveglianza e chiusura per il controllo",
    subtitle="% dei messaggi inviati dal gruppo che contengono il tema (parole chiave)"))
save(q2d, "q2d_temi_per_gruppo")

alt.FacetChart(...)

**Cosa si vede.** **Alias**: Nemo Reef (63%) e **soldi** (49%), più natura/tempo libero (il birdwatching di The Intern e The Lookout). **Operazione**: **permessi** (39%), **attrezzature/lavori** (36%), evento di Sailor Shift e segretezza. **Controllo**: Nemo Reef (64%), sorveglianza, ambiente e chiusura dell'area. **Investigatori**: ambiente, lavori e permessi, cioè seguono proprio l'operazione.

In [16]:
print("Controllo delle parole chiave: 2 esempi per tema (se un esempio non c'entra, il tema va corretto)")
rows = []
for t, p in TOPICS.items():
    for _, x in m[low.str.contains(p)].sample(2, random_state=1).iterrows():
        rows.append({"tema": t, "da": x.sender_clean, "testo": x.content})
display(pd.DataFrame(rows).style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))

Controllo delle parole chiave: 2 esempi per tema (se un esempio non c'entra, il tema va corretto)


tema,da,testo
Nemo Reef,Horizon,"Green Guardians, Horizon reporting. Have identified unauthorized deep-sea extraction equipment at Nemo Reef. Multiple executive personnel supervising from distance. Documenting evidence of container transfers between Neptune's team and unmarked civilian vessels. Requesting guidance on next monitoring steps."
Nemo Reef,Oceanus City Council,"Green Guardians, Oceanus City Council here. Your concerns about Nemo Reef have been noted, but the permit approval followed proper channels with senior authorization. Environmental monitoring will continue throughout operations. We'll schedule a briefing next week."
permessi,Small Fry,"Remora, Small Fry here. I confirm availability for 0530 Nemo Reef operation. I'll prepare specialized handling procedures for the fragile equipment via Route C and coordinate with the Glitters Team. Have you received final permit confirmation from the Boss?"
permessi,Horizon,"Green Guardians, Horizon reporting. Have identified unauthorized deep-sea extraction equipment at Nemo Reef. Multiple executive personnel supervising from distance. Documenting evidence of container transfers between Neptune's team and unmarked civilian vessels. Requesting guidance on next monitoring steps."
soldi / pagamenti,Mrs. Money,"Middleman, it's Mrs. Money. Conservation patrol activity has increased significantly around Nemo Reef. We may need to adjust our approach for tonight's meeting. I've identified alternative funding channels if needed. Still on for 7PM?"
soldi / pagamenti,The Intern,"Mrs. Money, The Intern reporting. The Lookout discovered specialized underwater concrete forms in those fragile crates. Harbor master mentioned 'deep water construction' multiple times. This suggests permanent structures at Nemo Reef, beyond equipment installation. How should I proceed?"
attrezzature / lavori,Neptune,Neptune to V. Miesel. Additional personnel received but still insufficient. Crews reporting extreme fatigue and equipment strain. Timeline pressure intensifying with council inquiries. Request executive decision on prioritizing critical path elements to meet deadline.
attrezzature / lavori,Reef Guardian,"Seawatch, Reef Guardian here. Will commence eastern quadrant patrol at 2100 hours. Have you identified vessel registration numbers for those delivering equipment? Need this documentation for our report. Our night vision systems are operational."
sorveglianza / pattuglie,The Lookout,The Lookout here! Those conservation vessels are still active - they're now filing formal protests against the reef closure. Saw them collecting water samples as evidence. Perfect day for osprey watching though! Still on for tomorrow's birdwatching session?
sorveglianza / pattuglie,Sentinel,"Green Guardians, Sentinel here. While docked for refueling, our crew overheard concerning information. The previously denied Nemo Reef permit is being expedited through approval. Someone with influence is pushing it. Requesting guidance on increasing our patrol schedule."


---
## Q3 — Pseudonimi

> *Alcune persone e barche usano pseudonimi (es. "Boss", "The Lookout"), e uno stesso pseudonimo può essere usato da più persone. Chi c'è dietro?*

Prima un controllo sui dati: **chi ha davvero inviato ogni messaggio** (metadati radio contro firma nel testo). Poi tre indizi sugli alias: **chi li nomina**, **quando sono attivi**, **con chi parlano**.

**Domanda.** Il mittente registrato dalla radio è quello vero?

**Come leggerlo.** Righe = mittente secondo i metadati radio, colonne = chi scrive davvero secondo la formula d'apertura del testo («Davis, Mako here» = Mako scrive a Davis). Il numero è quanti messaggi.

<sub>Codifica: mittente **N**, firma **N**, messaggi **Q** → matrice con valori scritti, scala magma da zero; tutti i 584 messaggi (anche quelli scartati dalla pulizia).</sub>

In [17]:
mis = m_all[m_all.sender_mismatch].groupby(["sender", "claimed_sender"]).size().reset_index(name="n")
mis = mis.set_axis(["mittente nel grafo", "firma nel testo", "n"], axis=1)
xo = mis.groupby("firma nel testo").n.sum().sort_values(ascending=False).index.tolist()
yo = mis.groupby("mittente nel grafo").n.sum().sort_values(ascending=False).index.tolist()
b = alt.Chart(mis).encode(x=alt.X("firma nel testo:N", sort=xo, axis=alt.Axis(labelAngle=-40)),
                          y=alt.Y("mittente nel grafo:N", sort=yo))
q3a = (b.mark_rect(cornerRadius=2).encode(color=alt.Color("n:Q", scale=alt.Scale(**SEQ, domainMin=0), legend=None),
                                          tooltip=["mittente nel grafo", "firma nel testo", "n"])
       + b.mark_text(fontSize=10).encode(text="n:Q", color=alt.condition("datum.n > 5", alt.value("white"), alt.value(INK)))
       ).properties(width=alt.Step(26), height=alt.Step(20), title=alt.Title(
    "La radio registra spesso il destinatario al posto del mittente",
    subtitle="messaggi in cui il mittente registrato è diverso da chi si presenta nel testo"))
save(q3a, "q3a_firme")

alt.LayerChart(...)

**Cosa si vede.** Non sono pseudonimi ma **errori dei metadati**: leggendo i testi, i messaggi «registrati come Davis, firmati Mako» iniziano con «Davis, Mako here», cioè **Mako scrive a Davis**. Dei 76 messaggi con mittente diverso (sui 581 senza duplicati), **43** hanno mittente e destinatario scambiati e **31** sono registrati come messaggi a sé stessi (es. Davis → Davis); solo **2** sono casi diversi. Conseguenza: le analisi usano il mittente corretto dal testo; anche alcune relazioni del grafo dello stagista (es. «Davis opera Mako») potrebbero derivare da questi errori.

In [18]:
print("Prova: messaggi registrati dalla radio come Davis → Davis, ma il testo dice «Davis, Mako here»")
show(m_all[(m_all.sender == "Davis") & (m_all.claimed_sender == "Mako")].sort_values("timestamp"),
     cols=("timestamp", "sender", "receiver", "content"), n=5)

Prova: messaggi registrati dalla radio come Davis → Davis, ma il testo dice «Davis, Mako here»


quando,sender,receiver,testo
01 Oct 09:02,Davis,Davis,"Davis, Mako here. Just completed documentation of our sites. Coral Point has excellent visibility but too exposed for filming. Dolphin Bay offers privacy but those currents will be challenging for equipment setup. Heading back to Himark as scheduled."
04 Oct 08:45,Davis,Davis,"Davis, Mako here. Preparing for departure from Himark Harbor. Executive client has arrived and joined the Sailors team. Will maintain minimal radio communication as instructed. Transponder active. Awaiting your final clearance."
04 Oct 11:11,Davis,Davis,"Davis, Mako here. Executive client is aboard and we're now outside Nemo Reef boundary. Transponder issue resolved but maintaining minimal communications as instructed. Sailors team conducting operations per protocol. Awaiting further instructions."
04 Oct 11:32,Davis,Davis,"Davis, Mako here. We've cleared Himark Harbor with the executive client aboard. Transponder active but maintaining radio silence as instructed. Sailors team is ready for operations. Position is secure. Awaiting further directives."
05 Oct 10:00,Davis,Himark Harbor,"Himark Harbor, Mako here. Safely docked at your location. Crew performing routine maintenance after yesterday's trip. Will require expedited departure clearance once our special permits arrive. Captain Davis expects documentation shortly from management."


**Domanda.** Chi parla *di* Boss, di Nadia e degli altri alias?

**Come leggerlo.** Righe = chi viene nominato, colonne = chi lo nomina in un messaggio (senza essere né mittente né destinatario). Il numero è quante volte.

<sub>Codifica: nominato **N**, chi nomina **N**, menzioni **Q** → matrice con valori scritti, righe e colonne ordinate per totale.</sub>

In [19]:
rows = []
for _, x in m.iterrows():
    for n in NAMES:
        if n not in (x.sender_clean, x.receiver_clean) and re.search(r"\b" + re.escape(n) + r"\b", x.content):
            rows.append((x.sender_clean, n, x.msg_id))
men = pd.DataFrame(rows, columns=["chi_nomina", "nominato", "msg_id"])
FOCUS = ["Boss", "The Middleman", "Mrs. Money", "The Accountant", "Small Fry", "The Lookout", "The Intern", "Nadia Conti"]
mf = men[men.nominato.isin(FOCUS)].groupby(["nominato", "chi_nomina"]).size().reset_index(name="n")
yo = mf.groupby("nominato").n.sum().sort_values(ascending=False).index.tolist()
xo = mf.groupby("chi_nomina").n.sum().sort_values(ascending=False).index.tolist()
b = alt.Chart(mf).encode(x=alt.X("chi_nomina:N", sort=xo, title="chi lo nomina", axis=alt.Axis(labelAngle=-40, orient="top")),
                         y=alt.Y("nominato:N", sort=yo, title="chi è nominato"))
q3b = (b.mark_rect(cornerRadius=2).encode(color=alt.Color("n:Q", scale=alt.Scale(**SEQ, domainMin=0), legend=None),
                                          tooltip=["nominato", "chi_nomina", "n"])
       + b.mark_text(fontSize=10).encode(text="n:Q", color=alt.condition("datum.n >= 4", alt.value("white"), alt.value(INK)))
       ).properties(width=alt.Step(26), height=alt.Step(24), title=alt.Title(
    "Boss è nominato da tutta l'operazione: è il loro riferimento",
    subtitle="quante volte un nome compare nel testo di messaggi di altri"))
save(q3b, "q3b_menzioni")

alt.LayerChart(...)

**Cosa si vede.** **Boss** è nominato da Mako, Remora, Neptune, Small Fry, Davis e dagli altri alias: è il capo comune dell'operazione. **The Lookout** è nominato quasi solo da The Intern. **Nadia** è nominata da Remora e Rodriguez; **The Middleman** e **The Accountant** nominano Nadia: un ponte diretto tra Nadia e gli alias.

In [20]:
print("Prova: messaggi che nominano Boss")
show(m[m.msg_id.isin(men[men.nominato == "Boss"].msg_id)], n=6)

Prova: messaggi che nominano Boss


quando,da,a,testo
01 Oct 09:52,Mrs. Money,The Middleman,"Middleman, it's Mrs. Money. I've prepared those financial projections we discussed. Particularly interested in your thoughts on that Nemo Reef loophole before my meeting with Boss tomorrow. Available to talk this evening?"
01 Oct 10:53,The Middleman,Mrs. Money,"Mrs. Money, this is The Middleman. I can meet this evening to discuss the Nemo Reef loophole before your meeting with Boss. Council just flagged increased vessel activity there - perfect timing for our plans. 7PM at the cafe?"
03 Oct 09:28,Remora,Rodriguez,"Rodriguez, Remora here. I've been scouting the southern coastline for alternative filming sites. When you're done with Mako, we should discuss talking to the Boss about Nemo Reef. Heard they have 'special arrangements' for certain clients."
04 Oct 08:12,Mrs. Money,The Middleman,"Middleman, Mrs. Money here. Meeting cancelled. Boss has ordered contingency plan B due to excessive conservation activity at Nemo Reef. Destroy all documentation. Will contact you through secure channel with new instructions."
04 Oct 10:21,The Middleman,Mrs. Money,"Mrs. Money, this is Middleman. I've redirected Council's attention to other areas. Meeting with Boss tonight at 9PM to discuss new focus. Have you secured those alternative funding channels you mentioned? Need status update."
05 Oct 10:05,Mako,Davis,"Captain Davis, Mako here. We're fully docked at Himark Harbor with maintenance underway. The crew is discussing yesterday's trip while preparing for quick departure once permits arrive. Harbor confirms our expedited clearance is approved. The Boss seems confident this time."


**Domanda.** Quando sono attivi gli alias? Due nomi si alternano nel tempo?

**Come leggerlo.** Ogni riga è un attore; ogni trattino è un suo messaggio (inviato o ricevuto) nel momento in cui è avvenuto. Il colore è il gruppo. Se un nome sparisce quando ne compare un altro, potrebbero essere la stessa persona.

<sub>Codifica: istante **T**, attore **N**, gruppo **N** → un segno per messaggio su asse temporale comune (posizione, la codifica più precisa); colore = gruppo.</sub>

In [21]:
LANES = ["Boss", "Nadia Conti", "The Middleman", "The Accountant", "Mrs. Money", "Davis", "Small Fry",
         "The Intern", "Sam", "The Lookout", "Kelly"]
lane = pd.concat([m[["timestamp", "sender_clean"]].set_axis(["timestamp", "attore"], axis=1),
                  m[["timestamp", "receiver_clean"]].set_axis(["timestamp", "attore"], axis=1)])
lane = lane[lane.attore.isin(LANES)].assign(gruppo=lambda d: d.attore.map(group))
ticks = alt.Chart(lane).mark_tick(thickness=2, size=14, opacity=0.85).encode(
    x=alt.X("timestamp:T", title=None, axis=alt.Axis(format="%d %b", tickCount=14)),
    y=alt.Y("attore:N", sort=LANES, title=None), color=color_group(), tooltip=["attore", "timestamp:T"])
q3c = (ticks + date_rules()).properties(width=620, height=alt.Step(22), title=alt.Title(
    "Boss è attivo soprattutto fino al 6 ottobre; Nadia entra in scena il 5",
    subtitle="un trattino per ogni messaggio inviato o ricevuto"))
save(q3c, "q3c_presenza_alias")

alt.LayerChart(...)

**Cosa si vede.** Boss è attivo dal 1 al 6 ottobre, poi ricompare solo l'11 e il 13; **Nadia entra il 5**. Le due presenze si toccano solo il 5–6: l'ipotesi **Boss = Nadia** è *plausibile* (indizio decisivo in q4b). The Accountant compare solo dall'11. The Intern, Sam e The Lookout hanno trattini quasi negli stessi istanti: si scrivono tra loro.

**Domanda.** Quali nomi hanno gli stessi interlocutori?

**Come leggerlo.** Righe = alias (più Davis, Rodriguez, Nadia); colonne = persone e barche. La cella è più scura quanto più i due parlano con *le stesse persone* (0 = nessun interlocutore in comune, 1 = identici). Due nomi con interlocutori quasi identici potrebbero essere la stessa persona.

<sub>Codifica: attore **N**, candidato **N**, similarità **Q** 0–1 → matrice con scala magma fissa 0–1 (serve a trovare valori alti, non a confrontarli al decimale). Nomi originali del grafo: lo pseudonimo è proprio il nome registrato.</sub>

In [22]:
und = pd.concat([m_all[["sender", "receiver"]].set_axis(["a", "b"], axis=1),
                 m_all[["receiver", "sender"]].set_axis(["a", "b"], axis=1)])
vec = und[und.a != und.b].groupby(["a", "b"]).size().unstack(fill_value=0)
vec = vec[vec.sum(axis=1) >= 5]
norm = vec.div((vec ** 2).sum(axis=1) ** 0.5, axis=0)
sim = (norm @ norm.T).rename_axis(index=None, columns=None)
alias = [a for a in act[act.is_alias_like].name if a in sim.index]
rows_ = alias + [p for p in ["Davis", "Rodriguez", "Nadia Conti"] if p in sim.index]
cols_ = [c for c in sim.columns if kind.get(c) in ("Person", "Vessel") and c not in rows_]
sl = sim.loc[rows_, cols_].stack().reset_index().set_axis(["attore", "candidato", "similarità"], axis=1)
sl = sl[sl.candidato.isin(sl.groupby("candidato")["similarità"].max().loc[lambda s: s >= 0.3].index)]
q3d = alt.Chart(sl).mark_rect(cornerRadius=2).encode(
    x=alt.X("candidato:N", title=None, sort=alt.EncodingSortField("similarità", op="max", order="descending"), axis=alt.Axis(labelAngle=-40)),
    y=alt.Y("attore:N", sort=rows_, title=None),
    color=alt.Color("similarità:Q", scale=alt.Scale(**SEQ, domain=[0, 1]), title="interlocutori in comune"),
    tooltip=["attore", "candidato", alt.Tooltip("similarità:Q", format=".2f")],
).properties(width=alt.Step(26), height=alt.Step(20), title=alt.Title(
    "The Intern parla con le stesse persone di Sam; Davis con quelle di Neptune e Mako",
    subtitle="somiglianza degli interlocutori (0 = nessuno in comune, 1 = identici); colonne con almeno un valore ≥ 0.3"))
save(q3d, "q3d_interlocutori_in_comune")

alt.Chart(...)

**Cosa si vede.** Gli alias principali (Boss, Mrs. Money, The Middleman) **non** somigliano a nessuno: gli interlocutori da soli non bastano. Un indizio: **The Intern ≈ Sam** (0.6). La somiglianza di **Davis con Neptune/Mako** dipende dagli errori dei metadati (q3a), non da un'identità condivisa. Per chiudere Q3 servono anche lo stile di scrittura e la lettura dei messaggi.

---
## Q4 — Nadia Conti

> *Nadia, in passato coinvolta in pesca illegale, sta continuando attività illecite a Oceanus? Le prove reggono?*

**Come rispondiamo.** La domanda chiede tre cose: (a) prove che Nadia fa, *o non fa*, qualcosa di illegale; (b) un riassunto visivo delle sue azioni; (c) se i sospetti di Clepper sono giustificati.
Il grafo da solo non basta: un arco dice *che* due attori si parlano, non *cosa* si dicono. Per questo abbiamo letto tutti i 37 messaggi di Nadia o su Nadia e i 25 del **Boss**,
e abbiamo classificato a mano quelli che descrivono un'azione (codifica manuale: ogni scelta è verificabile nella tabella sotto il grafico).

**Domanda.** Cosa fa Nadia, giorno per giorno? (riassunto delle azioni)

**Come leggerlo.** Ogni riga è un tipo di azione; ogni cerchio è un messaggio che la documenta, nel giorno in cui è stato scritto. Il numero rimanda alla tabella sotto. **Arancio** = Nadia; **rosa** = Boss, che è probabilmente Nadia (vedi il grafico successivo). **Cerchio pieno** = lo scrive o lo riceve lei; **cerchio vuoto** = altri ne parlano (prova indiretta, più debole).

<sub>Codifica: istante **T**, tipo di azione **N** (6, codifica manuale) → posizione su asse temporale comune e righe; chi agisce **N** → colore del gruppo (Nadia = Operazione, Boss = Alias); fonte diretta/indiretta **N** (2) → pieno/vuoto (l'incertezza si vede, S05); numeri come annotazioni che collegano grafico e testo (L17).</sub>

In [23]:
FILONI = ["ottiene i permessi", "paga chi deve controllare", "fa sparire o ripulisce documenti",
          "mente o depista i controlli", "segue i lavori a Nemo Reef", "copertura turistica"]
P, S, D, M, L, C = FILONI
# (n. messaggio, tipo di azione, chi agisce, fonte, sintesi) — codifica manuale leggendo i testi
EVID = [
    (21, P, "Boss", "diretta", "cerca una «scappatoia» nelle zone protette di Nemo Reef"),
    (249, D, "Boss", "diretta", "ordina di distruggere i documenti che collegano a Mako"),
    (312, C, "Boss", "diretta", "vuole documenti delle barche che mostrino «solo turismo»"),
    (328, P, "Nadia Conti", "indiretta", "Remora: «si occupa personalmente dei permessi per Nemo Reef»"),
    (331, D, "Nadia Conti", "diretta", "annulla il corridoio speciale e chiede di distruggere i documenti"),
    (334, M, "Nadia Conti", "diretta", "al Council: «non sapevo di approvazioni di permessi»"),
    (366, S, "Boss", "diretta", "chiede di mascherare le tracce dei soldi"),
    (388, P, "Nadia Conti", "indiretta", "Remora: «ha fatto firmare il commissario Torres in poche ore»"),
    (389, P, "Boss", "indiretta", "Glitters ringrazia Boss per i permessi accelerati con Torres"),
    (520, P, "Nadia Conti", "diretta", "lo staff di Sailor Shift la ringrazia per l'aiuto con i permessi"),
    (521, D, "Nadia Conti", "diretta", "a Davis: «creare subito una traccia di carte pulita»"),
    (529, S, "Nadia Conti", "diretta", "offre a Liam Thorne il doppio perché l'Harbor Master «collabori»"),
    (535, S, "Nadia Conti", "diretta", "Liam: pattuglie spostate a nostro favore, «porta il pagamento»"),
    (538, L, "Nadia Conti", "diretta", "rivede le attrezzature per le fondazioni: «restiamo sotto i radar»"),
    (584, M, "Nadia Conti", "diretta", "a Marlin: «deviazioni temporanee, niente di cui preoccuparsi»"),
    (588, P, "Nadia Conti", "indiretta", "Davis: «operiamo con il permesso #CR-7844», discrezione"),
    (601, L, "Nadia Conti", "diretta", "Elise: «costruzione subacquea permanente confermata»"),
    (739, M, "Boss", "diretta", "Liam: «ritarderò l'autorizzazione del ROV di EcoVigil»"),
    (753, M, "Nadia Conti", "diretta", "Liam: EcoVigil sospetta, bisogna ripulire le operazioni"),
    (795, M, "Nadia Conti", "diretta", "sposta l'attrezzatura: «EcoVigil la gestisco io»"),
    (847, P, "Nadia Conti", "diretta", "«documentazione per #CR-7844 completa», vede The Middleman"),
    (869, L, "Nadia Conti", "diretta", "ordina il trasporto finale di attrezzatura, protocolli cifrati"),
    (940, C, "Nadia Conti", "indiretta", "Glitters: «Nadia ha approvato tutto», autorizzazione Delta-3"),
    (943, C, "Nadia Conti", "diretta", "Remora: «facciata turistica mantenuta», chiede l'ok per le luci subacquee"),
]
ev4 = pd.DataFrame(EVID, columns=["n_msg", "azione", "attore", "fonte", "sintesi"]).assign(
    msg_id=lambda d: "Event_Communication_" + d.n_msg.astype(str))
ev4 = ev4.merge(m_all[["msg_id", "timestamp", "sender_clean", "receiver_clean", "content"]], on="msg_id")
ev4 = ev4.sort_values("timestamp").reset_index(drop=True)
ev4["#"] = ev4.index + 1
ev4["gruppo"] = ev4.attore.map(group)
ev4["k"] = ev4.groupby(["azione", ev4.timestamp.dt.date]).cumcount()  # affianca i messaggi dello stesso giorno

base = alt.Chart(ev4).encode(
    x=alt.X("timestamp:T", title=None, scale=alt.Scale(domain=["2040-10-01", "2040-10-15"]), axis=alt.Axis(format="%d %b", tickCount=14)),
    y=alt.Y("azione:N", sort=FILONI, title=None, axis=alt.Axis(labelLimit=230)), yOffset=alt.YOffset("k:O"))
pts = base.mark_point(size=150, strokeWidth=2, opacity=1).encode(
    color=alt.Color("gruppo:N", scale=GROUP_COLOR, legend=None),
    fill=alt.condition("datum.fonte == 'diretta'", alt.Fill("gruppo:N", scale=GROUP_COLOR, legend=None), alt.value("white")),
    tooltip=["#", "timestamp:T", "attore", "fonte", "sintesi"])
num = base.mark_text(align="left", dx=9, fontSize=10, color=INK).encode(text="#:N")
q4a = (pts + num + date_rules()).properties(width=640, height=alt.Step(18), title=alt.Title(
    "Nadia ottiene permessi con favori, paga chi controlla, ripulisce i documenti e depista le verifiche",
    subtitle=["arancio = Nadia, rosa = Boss (probabilmente Nadia); pieno = lo scrive o lo riceve, vuoto = altri ne parlano",
              "il numero rimanda alla tabella delle prove"]))
save(q4a, "q4a_azioni_nadia")

alt.LayerChart(...)

**Cosa si vede.** Le azioni **dirette** di Nadia (cerchi arancio pieni) coprono tutti e sei i tipi. Le più gravi: **12–13** (8 ott) paga **Liam Thorne** perché l'Harbor Master chiuda un occhio e Liam conferma di aver spostato le pattuglie; **5 e 11** chiede di distruggere documenti e di «creare una traccia di carte pulita»; **6** dice al Council di non sapere nulla dei permessi, mentre lo stesso giorno l'operazione racconta che li gestisce lei (**4**). Dal 10 ottobre il tono cambia: non procura più, **difende** (EcoVigil, ROV, microfoni subacquei). I punti rosa (Boss) anticipano le stesse azioni nella prima settimana.

In [24]:
print("Tabella delle prove: il numero è quello del grafico")
show(ev4.rename(columns={"#": "n"}), cols=("n", "timestamp", "sender_clean", "receiver_clean", "azione", "fonte", "content"))

Tabella delle prove: il numero è quello del grafico


n,quando,da,a,azione,fonte,testo
1,01 Oct 09:49,Boss,Mrs. Money,ottiene i permessi,diretta,"Mrs. Money, this is Boss. After speaking with The Middleman, I'm confident about our Nemo Reef opportunity. He mentioned a potential loophole in protection zones. Let's discuss this during our 3PM meeting along with funding allocations."
2,04 Oct 08:10,Boss,Mrs. Money,fa sparire o ripulisce documenti,diretta,"Mrs. Money, this is Boss. Abort all Nemo Reef operations immediately. Conservation vessels are too active in the area. Implement contingency plan B - redirect funds through secondary channels and have The Intern destroy any documentation linking us to Mako's activities."
3,04 Oct 13:48,Boss,V. Miesel Shipping,copertura turistica,diretta,This is Boss. Need urgent confirmation that assets are properly reallocated to our tourism ventures. Conservation scrutiny has increased dramatically. Ensure all vessel documentation shows legitimate tourist operations only. Expecting verification by tomorrow.
4,05 Oct 09:00,Remora,Rodriguez,ottiene i permessi,indiretta,"Rodriguez, Remora here. Overheard crew saying Davis' meeting with Nadia Conti went well - she's personally handling permits for Nemo Reef with approval expected today. Should we prepare for deployment? Need your instructions."
5,05 Oct 09:45,Nadia Conti,Haacklee Harbor,fa sparire o ripulisce documenti,diretta,"Haacklee Harbor, this is Nadia Conti. I need to cancel the special access corridor arrangements for Nemo Reef immediately. Plans have changed due to unforeseen circumstances. Destroy all related documentation. I'll contact you when we're ready to proceed with alternative locations."
6,05 Oct 09:49,Nadia Conti,Oceanus City Council,mente o depista i controlli,diretta,This is Nadia Conti. My cancellation was due to scheduling conflicts with our tourism development initiatives. I wasn't aware of any permit approvals. I'll submit revised documentation for alternative sustainable tourism proposals next week.
7,05 Oct 10:38,Boss,Mrs. Money,paga chi deve controllare,diretta,"Mrs. Money, Boss here. Need you to verify all financial trails are properly disguised through the tourism ventures. Destroy any documentation linking us to Nemo Reef operations. City Council is asking questions. Meet me at the usual spot tonight."
8,06 Oct 09:57,Remora,Sailor Shifts Team,ottiene i permessi,indiretta,Remora here to Sailor Shifts. We'll need additional event staff aboard tomorrow. Davis mentioned thanking Nadia Conti personally - she called in favors at the permit office and got Commissioner Torres to sign off within hours. Unprecedented speed.
9,06 Oct 09:58,Glitters Team,Boss,ottiene i permessi,indiretta,"Hi Boss, Glitters Team here. We heard about your assistance with the permits - truly appreciate you expediting things with Commissioner Torres! We'll need to discuss staffing for tomorrow's setup. When can we connect?"
10,07 Oct 11:57,Sailor Shifts Team,Nadia Conti,ottiene i permessi,diretta,"Hi Nadia, this is the Sailor Shifts Team. Received your message about permit assistance - thank you! We urgently need to discuss tomorrow's staffing requirements. Can you confirm how many additional crew members we should bring for the setup?"


**Domanda.** Il Boss è Nadia? (serve a sapere se le azioni del Boss sono sue)

**Come leggerlo.** Ogni riga è un indizio: una stessa cosa che riguarda sia Boss sia Nadia. Il punto rosa è il messaggio su Boss, quello arancio il messaggio su Nadia; la linea li unisce e il numero dice quanto tempo passa tra i due. Linea corta = la stessa notizia arriva a «due persone» quasi insieme.

<sub>Codifica: indizio **N**, istante **T** → dumbbell (due punti su asse temporale comune, distanza = intervallo); attore **N** → colore del gruppo (Boss = Alias, Nadia = Operazione).</sub>

In [25]:
PAIRS = [("ringraziati per i permessi firmati da Torres", 389, 388),
         ("Liam Thorne avvisa del ROV di EcoVigil", 739, 753),
         ("chiedono di distruggere documenti", 249, 331),
         ("lavorano con The Accountant", 726, 538),
         ("incontrano The Middleman", 11, 847)]
ts = m_all.set_index(m_all.msg_id.str.split("_").str[-1].astype(int)).timestamp
pr = pd.DataFrame([(lab, ts[b], ts[n]) for lab, b, n in PAIRS], columns=["indizio", "Boss", "Nadia Conti"])
gap = (pr["Nadia Conti"] - pr.Boss).abs()
pr["distanza"] = [f"{int(g.total_seconds() // 60)} min" if g < pd.Timedelta("1h") else f"{g.days} giorni" if g.days > 1 else f"{g.days} giorno"
                  for g in gap]
pr["fine"] = pr[["Boss", "Nadia Conti"]].max(axis=1)
pr["riga"] = range(len(pr))
pl = pr.melt(id_vars=["indizio", "riga"], value_vars=["Boss", "Nadia Conti"], var_name="attore", value_name="quando")
pl["pos"] = pl.riga + pl.attore.map({"Boss": -0.17, "Nadia Conti": 0.17})  # Boss sopra, Nadia sotto: visibili anche a 1 minuto
YS = alt.Scale(domain=[-0.6, len(pr) - 0.4], reverse=True, nice=False)
y = alt.Y("pos:Q", title=None, axis=None, scale=YS)
x = alt.X("quando:T", title=None, scale=alt.Scale(domain=["2040-10-01", "2040-10-15"]), axis=alt.Axis(format="%d %b", tickCount=14))
line = alt.Chart(pl).mark_line(color="#bbbbbb", strokeWidth=2).encode(x=x, y=y, detail="indizio:N")
dots = alt.Chart(pl).mark_point(filled=True, size=160, opacity=1).encode(
    x=x, y=y, color=alt.Color("attore:N", scale=alt.Scale(domain=["Boss", "Nadia Conti"], range=["#CC79A7", "#D55E00"]), title=None),
    tooltip=["indizio", "attore", "quando:T"])
lab = alt.Chart(pr).mark_text(align="left", dx=12, fontSize=11, color=INK).encode(
    x=alt.X("fine:T"), y=alt.Y("riga:Q", scale=YS, axis=None), text="distanza:N")
rows_lab = alt.Chart(pr).mark_text(align="right", fontSize=11, color=INK).encode(
    x=alt.value(250), y=alt.Y("riga:Q", scale=YS, axis=None), text="indizio:N").properties(width=250, height=230)
q4b = alt.hconcat(rows_lab, (line + dots + lab).properties(width=560, height=230), spacing=10).properties(title=alt.Title(
    "Boss e Nadia ricevono lo stesso ringraziamento a 1 minuto di distanza: probabilmente sono la stessa persona",
    subtitle="stessa notizia o stesso contatto per Boss (rosa, sopra) e per Nadia (arancio, sotto); il numero è il tempo tra i due messaggi"))
save(q4b, "q4b_boss_e_nadia")

alt.HConcatChart(...)

**Cosa si vede.** Il 6 ottobre alle 09:57 Remora scrive che **Nadia** ha fatto firmare i permessi al commissario Torres; alle 09:58 Glitters ringrazia **Boss** per *lo stesso favore*. L'11 ottobre Liam Thorne avvisa del ROV di EcoVigil prima Boss (09:07) e poi Nadia (09:20). Entrambi chiedono di distruggere documenti, lavorano con The Accountant e incontrano The Middleman; nel grafo entrambi hanno **accesso speciale a Nemo Reef**. Inoltre, da quando Nadia compare (5 ott) Boss **non scrive più**, riceve soltanto. Conclusione: **Boss = Nadia è molto probabile**, ma resta un'inferenza; per questo nel grafico precedente i punti del Boss hanno un colore diverso e il verdetto non dipende da loro.

**Verdetto (c): i sospetti di Clepper sono giustificati, ma non per la pesca.**

| Accusa | Prove (numeri del grafico q4a) | Cosa la indebolisce | Forza |
|---|---|---|---|
| **Corruzione**: paga Liam Thorne perché l'Harbor Master collabori | 12, 13; effetti in 18–20 (Liam devia EcoVigil) | nulla: lo scrive lei | forte |
| **Distruzione / falsificazione di documenti** | 5, 11 (e 2 se Boss = Nadia) | il 5 ott si giustifica con «conflitti di calendario» (6), ma il 7 parla di «carte pulite» (11) | forte |
| **Mente alle autorità** | 6 contro 4 (stesso giorno); 15 a Marlin | 4 è un «sentito dire» della ciurma | media-forte |
| **Permesso #CR-7844 ottenuto con favori** | 4, 8, 10, 16, 21 (e 9) | il permesso è firmato da un commissario, quindi formalmente valido; nessun messaggio di Torres | media |
| **Lavori in area protetta** | 14, 17, 22 | l'8 ott Nadia stessa contesta i «lavori di fondazione fuori dagli accordi» (msg 536): l'idea sembra di V. Miesel / Neptune, ma lei continua a coprirli | forte sul coinvolgimento, media sul ruolo di capo |
| **Pesca illegale** (il sospetto di partenza) | nessun messaggio | — | nessuna prova |

**Prove che *non* fa nulla di illegale?** Solo la sua versione al Council (6), smentita dai suoi stessi messaggi successivi. L'evento di Sailor Shift è reale, ma i messaggi 23–24 lo chiamano «facciata».

**Domanda.** Contesto: cosa succede in mare a Nemo Reef mentre Nadia gestisce i permessi?

**Come leggerlo.** Asse orizzontale = tempo; ogni riga è una barca. Ogni simbolo è uno spostamento verso Nemo Reef o le secche vicine: ▲ incontro in mare (*rendezvous*), ● partenza, ■ tipo non indicato. Colore = gruppo.

<sub>Codifica: istante **T**, barca **N**, tipo di spostamento **N** (3) → timeline; forma = tipo di spostamento, colore = gruppo.</sub>

In [26]:
vm = ee[ee.event_type == "VesselMovement"].merge(ev[["id", "movement_type", "destination"]], left_on="event_id", right_on="id")
vm = vm[vm.entity_id.map(kind).eq("Vessel") & vm.destination.fillna("").str.contains("Nemo|reef|shoal", case=False)].dropna(subset=["timestamp"])
vm["spostamento"] = vm.movement_type.map({"rendezvous": "incontro in mare", "departure": "partenza"}).fillna("non indicato")
vm["gruppo"] = vm.entity_id.map(group)
q4c = (alt.Chart(vm).mark_point(filled=True, size=120, opacity=0.9).encode(
    x=alt.X("timestamp:T", title=None, scale=alt.Scale(domain=["2040-10-01", "2040-10-15"]), axis=alt.Axis(format="%d %b", tickCount=14)),
    y=alt.Y("entity_id:N", sort=alt.EncodingSortField("timestamp", op="min"), title=None),
    color=color_group(),
    shape=alt.Shape("spostamento:N", scale=alt.Scale(domain=["incontro in mare", "partenza", "non indicato"],
                                                     range=["triangle-up", "circle", "square"]), title="spostamento"),
    tooltip=["entity_id", "timestamp:T", "spostamento", "destination"])
    + date_rules()).properties(width=560, height=alt.Step(24), title=alt.Title(
    "L'8–9 ottobre, mentre Nadia paga Liam, le barche dell'operazione si incontrano a Nemo Reef",
    subtitle="spostamenti di barche verso Nemo Reef o le secche vicine"))
save(q4c, "q4c_barche_nemo_reef")

alt.LayerChart(...)

**Cosa si vede.** Gli **incontri in mare** di Mako, Remora e Neptune cadono l'**8–9 ottobre**, gli stessi giorni dei messaggi 12–17 (pagamento a Liam, attrezzature pesanti, costruzione permanente). Le barche di controllo (Reef Guardian, Sentinel, EcoVigil) partono verso la zona negli stessi giorni: i controlli c'erano, ed è per questo che Nadia li depista.

---
## Riepilogo delle ipotesi emerse

| Domanda | Ipotesi | Forza dell'evidenza |
|---|---|---|
| Q1 | La radio si usa 8–14, le attività avvengono all'alba e di sera | forte (q1a, q1c) |
| Q1 | Il ritmo cambia in W2, quando partono i lavori | media (q1b) |
| Q1c | Davis mette in moto Nadia; tra le barche gli ordini passano di barca in barca | indizio (q1f) |
| Q2 | 4 gruppi: operazione, controllo, alias, investigatori; gli alias finanziano l'operazione | forte (q2a, q2b, q2d) |
| Q3 | La radio scambia spesso mittente e destinatario (74 messaggi): va corretto con il testo prima di ogni analisi | forte (q3a) |
| Q3 | Boss è il capo dell'operazione; Boss = Nadia (stesso favore ringraziato a 1 minuto di distanza) | forte indizio (q3b, q3c, q4b) |
| Q3 | The Intern e Sam molto vicini | indizio (q3d) |
| Q4 | Nadia corrompe Liam Thorne, fa sparire documenti, mente al Council e depista i controlli | forte: sono messaggi suoi (q4a) |
| Q4 | L'evento di Sailor Shift fa da copertura ai lavori a Nemo Reef | forte (q4a: «tourism facade») |
| Q4 | I lavori permanenti sono un'idea di V. Miesel / Neptune, non di Nadia | da verificare (msg 536, 601) |
| Q4 | Pesca illegale | nessuna prova |

Il passo successivo è l'**analisi**: verificare le ipotesi "media" e "indizio" leggendo i messaggi.